# Deposit Attrition EDA — v4

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v3 fixed how the event study was *measured*. v4 fixes what it was measured *on*,
and turns the output from a set of univariate thresholds into a ranked worklist.

It reads the v2-shaped panels (repointed at the 2023-extended rebuild) and adds
nothing from outside them.

**What changed and why**

| | v3 | v4 |
|---|---|---|
| `DATE_START` / `DATE_END` | declared, never referenced | panel span is **read from the data** and asserted; the config values are advisory only |
| new-entity burn-in | three hard-coded months blanked | `cpty_new_out` / `fin_new_out` **rebuilt from `pay_pairs` on a trailing 12-month lookback**. Stationary across calendar time; no burn-in hack. Old and new compared side by side |
| lead search | `SEARCH_FROM = -9`, and the two earliest features separated **at** -9 | two horizon configs (`H12`, `H18`); a separation landing on `SEARCH_FROM` is reported as **CENSORED**, not as a result |
| control cohort | `event_m` drawn with `limit(2000)` | proper draw from the empirical event-month distribution |
| monthly hazard | events / (all customers × all evaluable months) | events / **at-risk** customer-months. The old denominator counts customers who already left |
| `A_full_exit` at the panel edge | could fire on 1 month of confirmation | `A_REQUIRE_FWD` — the last months are censored, not events |
| `segment_desc` | `F.max()` over the whole history | value at **entry**, plus a QA block measuring how often segment is restated in the run-up to an event |
| labels | A and B | A, B, **and `AB_union`** (§8.5) |
| features | 21 raw levels | + **peer percentile ranks** (NAICS2 × size-decile × month, with backoff) and + **trend block** (3/6m ratios, YoY, decline counts, months since peak) |
| operating points | 8 features, `rel_m` -12…-1 | all features **including `cpty_new_out` / `fin_new_out`** (§7.1), restricted to `rel_m ≤ -9` (§7.2), captions corrected (§7.3) |
| the deliverable | thresholds | **precision@k on a ranked worklist**, by horizon, by calendar period, and dollar-weighted (§8.1, §8.6) |
| output | tables | + `model_matrix` parquet, the input to the hazard model |

**Nothing new is pulled.** Peer groups, tenure and segment come from columns
already on the deposit panel (`cust_naics_cd_val`, `segment_desc`, `market_desc`,
`state`, `opened_dt`). The rebuilt new-entity features come from
`attrition_v2/pay_pairs`, which v2 already wrote. Every optional column is
resolved by a schema probe in §0 and skipped with a loud warning if absent.

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v4
# =====================================================================
from pathlib import Path

DB           = "dsihd01p_dsi"
TBL_DEPOSITS = f"{DB}.lap_dsi_universe_optimized"

# ADVISORY ONLY. v4 reads panels, it does not build them. The true span is
# read from the data in §1 and printed next to these. They are here so the
# notebook records which build it was pointed at, not to control anything.
DATE_START   = "2023-01-01"
DATE_END     = "2026-08-31"

# ── Panel source ──────────────────────────────────────────────────────
# Point this at the 2023-extended rebuild. Do NOT overwrite attrition_v2 -
# the briefed v3 numbers have to stay reproducible, and an A/B against the
# old build is the only way to tell a changed conclusion from a broken one.
PANELS       = "hdfs://nameservice1/user/pk36814/attrition_2023"
PANELS_PREV  = "hdfs://nameservice1/user/pk36814/attrition_v2"   # for the A/B, optional
HDFS_DIR     = "hdfs://nameservice1/user/pk36814/attrition_v4"
OUT_DIR      = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v4")

MAX_ROWS     = 60
ZERO_TOL     = 1.0
SEED         = 20260905

# Write the feature panel to parquet between blocks instead of stacking ~70
# window operations into one plan. Costs two writes, buys a run that finishes.
MATERIALISE_PANEL = True

# ── Labels ────────────────────────────────────────────────────────────
MIN_HIST_M      = 12       # keep at 12 - raising it discards the positives
MIN_LIVE_BEFORE = 6        # the 2023 pull was meant to recover
BAL_EXIT_FRAC   = 0.05
BAL_EXIT_HOLD   = 3
A_REQUIRE_FWD   = 3        # NEW. A_full_exit needs 3 forward months of
                           # confirmation like B does. Without it an event can
                           # fire on the final panel month with obs_fwd = 1.
LABEL_DEFS      = ["A_full_exit", "B_bal_exit", "AB_union"]
STUDY_DEFS      = ["A_full_exit", "B_bal_exit"]

# ── New-entity rebuild (replaces BURN_IN_YM entirely) ─────────────────
# cpty_new_out counted "counterparty not seen since the panel started". That
# set only grows, so the feature decays across calendar time for the whole
# panel - 12.4 in month 1 against ~0.5 later is a decay curve, not a
# three-month cliff. A feature that encodes calendar position will read
# differently in every rolling-origin fold. Redefine it on a fixed trailing
# window and the drift goes away; the first LOOKBACK_M months become warm-up
# and are nulled by construction, not by a hard-coded list.
REBUILD_NEW_ENTITY = True
LOOKBACK_M         = 12
PAIRS_DIR          = "pay_pairs"
# Column names inside pay_pairs are resolved by probe; override if the probe
# guesses wrong. entity_kind distinguishes counterparty rows from institution
# rows; leave as None to auto-resolve.
PAIRS_KIND_COL     = None
PAIRS_ENTITY_COL   = None

# ── Event study — two horizon configurations ──────────────────────────
# The v3 headline rests on cpty_new_out separating at rel_m -9, which IS
# SEARCH_FROM. The measurement is right-censored: the true lead may be 12 or
# 15 months and the search could not see it. H18 is the censoring test.
# EVENT_PRE also gates cohort membership, so H18 buys reach at the cost of
# cohort size. Run both.
CONFIGS = {
    "H12": dict(EVENT_PRE=12, BASE_WINDOW=(-12, -10), SEARCH_FROM=-9),
    "H18": dict(EVENT_PRE=18, BASE_WINDOW=(-18, -16), SEARCH_FROM=-15),
    # "H24": dict(EVENT_PRE=24, BASE_WINDOW=(-24, -22), SEARCH_FROM=-21),
}
RUN_CONFIGS  = ["H12", "H18"]
PRIMARY_CFG  = "H12"          # the cohort-maximising run; H18 is the reach test
EVENT_POST   = 3

# Seasonal baseline. The (-12,-10) window mixes one seasonally-matched month
# (-12 is the same calendar month as the event) with two that are not. With
# 18+ months of pre-window an anchored baseline is available.
BASE_MODE     = "window"      # "window" | "seasonal"
BASE_ANCHORS  = [-12, -24]    # used when BASE_MODE == "seasonal"

MIN_CELL_N   = 200
SEP_LEVEL    = 0.15
SEP_SHARE    = 0.03
SEP_RATE     = 0.05
HOLD         = 2

# ── Operating points ──────────────────────────────────────────────────
OP_THRESH    = [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]
OP_MAX_REL_M = -9          # FIX v3 §7.2 - rel_m > -9 sits inside the baseline
OP_ZERO_RULE = True        # also score the "feature fell to exactly zero" cut,
                           # which is the only usable cut for sparse counts
TARGET_PREC  = 0.25
MIN_RECALL   = 0.20

# ── Peer normalisation (§8.3 row 1 - the largest FP lever) ────────────
# Finest cell first; a customer-month falls back to the next level when its
# cell is thinner than MIN_PEER_N. All keys are columns already on the panel.
PEER_LEVELS  = [["naics2", "size_dec"], ["size_dec"], []]
MIN_PEER_N   = 50
N_SIZE_DEC   = 10

# ── Trend block (§8.3 row 4) ──────────────────────────────────────────
TREND_FEATS   = ["bal_live", "amt_out", "amt_in", "n_out", "n_in",
                 "cpty_out_n", "fin_out_n"]
TREND_WINDOWS = [3, 6]
PEAK_WINDOW   = 12
YOY           = True        # needs 13+ months of panel; checked in §1

# ── Ranked worklist (§8.1) ────────────────────────────────────────────
HORIZONS     = [1, 3, 6]
TOPK         = [100, 250, 500, 1000, 2500]
ATTR_LAG_M   = 6            # static attributes are read as of m - ATTR_LAG_M
                            # so a restatement made during the run-up cannot
                            # leak into the score
WORKLIST_DEF = "A_full_exit"

# ── Feature taxonomy ──────────────────────────────────────────────────
DENSE  = ["bal_live", "amt_out", "amt_in", "net_flow", "n_out", "n_in",
          "cpty_out_n", "fin_out_n"]
SPARSE = ["share_out_selfpay", "cpty_new_out", "fin_new_out",
          "share_out_check", "share_out_wire", "share_out_origpnc",
          "share_out_internal"]
MECHANIC   = ["cpty_out_hhi", "cpty_out_top", "fin_out_hhi", "fin_out_top"]
SHARE_LIKE = ["share_out_ach", "share_out_card"]

# DROP the four mechanical features from the feature set (v3 §6.9 held the
# counterparty count fixed and the gap vanished). They stay in MECHANIC so
# §9 can re-prove it on the longer panel, and are excluded everywhere else.
DROP_MECHANIC = True

# Direction: -1 means a LOW value is the risk signal. Used to build the score.
RISK_DIRECTION = {
    "bal_live": -1, "amt_out": -1, "amt_in": -1, "net_flow": -1,
    "n_out": -1, "n_in": -1, "cpty_out_n": -1, "fin_out_n": -1,
    "cpty_new_out": -1, "fin_new_out": -1,
    "share_out_internal": -1, "share_out_origpnc": -1,
    "share_out_selfpay": +1,
}

In [ ]:
# =====================================================================
# 1 · IMPORTS, HELPERS, SCHEMA PROBE
# =====================================================================
import warnings, datetime as dt, json, math
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

spark = (SparkSession.builder
         .appName("pkg_attrition_eda_v4")
         .config("spark.sql.shuffle.partitions", "400")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .enableHiveSupport().getOrCreate())

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HDFS URIs are plain strings, never pathlib - Path collapses hdfs:// to hdfs:/
def hp(name): return f"{HDFS_DIR.rstrip('/')}/{name}"
def sp(name): return f"{PANELS.rstrip('/')}/{name}"

_FIND = OUT_DIR / "FINDINGS_v4.csv"
FINDINGS = pd.read_csv(_FIND).to_dict("records") if _FIND.exists() else []
WARNINGS = []

def note(qid, question, answer, detail=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=question, answer=str(answer), detail=str(detail)))
    pd.DataFrame(FINDINGS).to_csv(_FIND, index=False)

def warn(msg):
    WARNINGS.append(msg)
    print(f"!! {msg}")

def _dec_safe(sdf):
    out = sdf
    for n, t in sdf.dtypes:
        if t.startswith("decimal"):
            out = out.withColumn(n, F.col(n).cast("double"))
    return out

def disp(obj, title=None, n=None, save=None, transpose=False):
    n = MAX_ROWS if n is None else n
    out = _dec_safe(obj).limit(n).toPandas() if hasattr(obj, "toPandas") else (
        obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(d, title=None, save=None):
    return disp(pd.DataFrame({"metric": list(d.keys()), "value": list(d.values())}),
                title=title, n=len(d), save=save)

def pct(a, b): return float(a) / float(b) if b else float("nan")

def resolve(df, candidates, what, required=False):
    """First candidate column present, case-insensitively. None if absent."""
    low = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    msg = f"{what}: none of {candidates} found in {sorted(df.columns)[:25]}..."
    if required:
        raise KeyError(msg)
    warn(msg)
    return None

def median_w(col, w):
    """Median over a Spark window. percentile_approx is not a window fn."""
    a = F.array_sort(F.collect_list(col).over(w))
    return F.element_at(a, (F.size(a) / 2).cast("int") + 1)

# ── load panels ───────────────────────────────────────────────────────
acct_month = spark.read.parquet(sp("panel_account_month")).persist(StorageLevel.DISK_ONLY)
cust_month = spark.read.parquet(sp("panel_customer_month")).persist(StorageLevel.DISK_ONLY)
feat0      = spark.read.parquet(sp("panel_pay_features")).persist(StorageLevel.DISK_ONLY)

# ── schema probe. Every optional column used later is resolved HERE, once,
#    so a missing column is a printed warning at the top of the run rather
#    than a NameError forty minutes in.
SCHEMA = pd.concat([
    pd.DataFrame({"panel": p, "column": [c for c, _ in d.dtypes],
                  "dtype": [t for _, t in d.dtypes]})
    for p, d in [("acct_month", acct_month), ("cust_month", cust_month),
                 ("pay_features", feat0)]])
disp(SCHEMA, title="0a &middot; Panel schemas (the probe every optional column is resolved against)",
     n=400, save="v4_schema")

COL = {}
COL["naics"]   = resolve(cust_month, ["cust_naics_cd_val", "naics_cd", "naics"], "NAICS on cust_month")
if COL["naics"] is None:
    COL["naics_acct"] = resolve(acct_month, ["cust_naics_cd_val", "naics_cd"], "NAICS on acct_month")
COL["segment"] = resolve(cust_month, ["segment_desc", "segment"], "segment_desc")
COL["market"]  = resolve(cust_month, ["market_desc", "market"], "market_desc")
COL["state"]   = resolve(cust_month, ["state", "state_cd"], "state")
COL["opened"]  = resolve(acct_month, ["opened_dt", "acct_opened_dt"], "opened_dt (tenure)")
COL["balmin"]  = resolve(cust_month, ["bal_min"], "bal_min")
COL["balmax"]  = resolve(cust_month, ["bal_max"], "bal_max")
COL["nlive"]   = resolve(cust_month, ["n_accts_live"], "n_accts_live", required=True)
COL["nacct"]   = resolve(cust_month, ["n_accts"], "n_accts")
COL["allcl"]   = resolve(cust_month, ["all_closed"], "all_closed", required=True)

# ── id dtype guard. The 2026-07 incident was an int64/str mismatch that
#    produced entirely plausible output.
for nm, d in [("cust_month", cust_month), ("pay_features", feat0)]:
    t = dict(d.dtypes).get("cust_pwr_id")
    assert t == "string", f"{nm}.cust_pwr_id is {t}, must be string"

ALL_FEATS = [f for f in (DENSE + SPARSE + ([] if DROP_MECHANIC else MECHANIC) + SHARE_LIKE)
             if f in feat0.columns or f == "bal_live"]
print("features carried:", len(ALL_FEATS), ALL_FEATS)
print("dropped as mechanical (v3 6.9):", MECHANIC if DROP_MECHANIC else "none")

## 1 · Panel provenance, span, and the stationarity audit

In [ ]:
# =====================================================================
# 2 · PANEL PROVENANCE AND THE NEW-ENTITY STATIONARITY AUDIT
#                                                       [OUTPUT BLOCK 1]
# =====================================================================
# Two jobs.
#
# (a) Establish the span FROM THE DATA. In v3 DATE_START and DATE_END were
#     declared and never referenced, so the config could say one thing while
#     the parquet said another and nothing would complain.
#
# (b) Show why BURN_IN_YM had to go. If the mean of cpty_new_out falls
#     monotonically across CALENDAR time - not just for three months - then
#     the feature encodes panel position, blanking three months does not fix
#     it, and every rolling-origin fold will read the feature differently.

_span = (cust_month.agg(F.min("ym").alias("ym_min"), F.max("ym").alias("ym_max"),
                        F.min("m_idx").alias("m_min"), F.max("m_idx").alias("m_max"),
                        F.countDistinct("m_idx").alias("n_months"),
                        F.countDistinct("cust_pwr_id").alias("n_cust"),
                        F.count("*").alias("n_rows")).collect()[0].asDict())
M_MIN, M_MAX = int(_span["m_min"]), int(_span["m_max"])
N_MONTHS     = int(_span["n_months"])
YM_MIN, YM_MAX = _span["ym_min"], _span["ym_max"]

_cfg_start = DATE_START[:7]
_cfg_end   = DATE_END[:7]
if YM_MIN != _cfg_start or YM_MAX != _cfg_end:
    warn(f"config says {_cfg_start}..{_cfg_end}; the panel is {YM_MIN}..{YM_MAX}. "
         f"Everything below uses the PANEL. Fix the config or the pull.")

_max_pre = max(CONFIGS[c]["EVENT_PRE"] for c in RUN_CONFIGS)
if N_MONTHS < _max_pre + MIN_HIST_M + BAL_EXIT_HOLD + 6:
    warn(f"{N_MONTHS} months is thin for EVENT_PRE={_max_pre}. Expect a small "
         f"H18 cohort; read 5b before trusting the censoring test.")
YOY_OK = YOY and N_MONTHS >= 13
if YOY and not YOY_OK:
    warn("panel shorter than 13 months - YoY features disabled")

kv({"panel path": PANELS,
    "ym span (from data)": f"{YM_MIN} .. {YM_MAX}",
    "m_idx span": f"{M_MIN} .. {M_MAX}",
    "months": N_MONTHS,
    "customers": f"{_span['n_cust']:,}",
    "customer-months (deposits)": f"{_span['n_rows']:,}",
    "customer-months (payments)": f"{feat0.count():,}",
    "account-months": f"{acct_month.count():,}",
    "config DATE_START/END (advisory)": f"{DATE_START} .. {DATE_END}",
    "YoY features": "on" if YOY_OK else "OFF",
    "configs to run": ", ".join(RUN_CONFIGS)},
   title="1a &middot; Panel provenance — span read from the data, not the config",
   save="v4_panel_span")

# ── (b) calendar-time drift of the v2 new-entity definition ───────────
NEW_ENTITY = [c for c in ("cpty_new_out", "fin_new_out") if c in feat0.columns]
if NEW_ENTITY:
    drift = (feat0.groupBy("ym").agg(
                F.count("*").alias("n"),
                *[F.avg(F.col(c).cast("double")).alias(f"mean_{c}") for c in NEW_ENTITY],
                *[F.avg((F.col(c) > 0).cast("double")).alias(f"rate_{c}") for c in NEW_ENTITY])
             .orderBy("ym")).toPandas()
    disp(drift, title="1b &middot; v2 new-entity definition against CALENDAR time. A three-month "
                      "burn-in would show a cliff then a flat line; a cumulative-history "
                      "definition shows a decay that never stops",
         n=60, save="v4_new_entity_drift")

    _d = drift.dropna(subset=[f"mean_{NEW_ENTITY[0]}"])
    _first, _last = _d.head(3), _d.tail(6)
    _mid = _d.iloc[3:-6] if len(_d) > 12 else _d
    _rows = []
    for c in NEW_ENTITY:
        f3, m_, l6 = _first[f"mean_{c}"].mean(), _mid[f"mean_{c}"].mean(), _last[f"mean_{c}"].mean()
        _rows.append(dict(feature=c, mean_first_3m=round(f3, 3), mean_middle=round(m_, 3),
                          mean_last_6m=round(l6, 3),
                          first3_over_middle=round(pct(f3, m_), 2),
                          middle_over_last6=round(pct(m_, l6), 2),
                          cv_across_months=round(_d[f"mean_{c}"].std() / _d[f"mean_{c}"].mean(), 3)))
    _dr = pd.DataFrame(_rows)
    disp(_dr, title="1c &middot; Is it a burn-in or a trend? first3/middle &gt;&gt; 1 is the burn-in. "
                    "middle/last6 also &gt; 1 means the drift continues and blanking cannot fix it",
         save="v4_new_entity_drift_summary")
    _cont = float(_dr["middle_over_last6"].max())
    note("DRIFT", "Is the v2 new-entity burn-in really only three months?",
         ("NO - the decay continues past the burn-in "
          f"(middle/last6 = {_cont:.2f})" if _cont > 1.15 else
          f"Mostly yes (middle/last6 = {_cont:.2f})"),
         "A cumulative 'not seen since panel start' set only grows, so the new-entity rate "
         "falls with panel age. Blanking three months removes the cliff and leaves the trend, "
         "which is what breaks an out-of-time fold. Section 2 rebuilds on a trailing window.")

## 2 · New-entity features rebuilt on a trailing window — `BURN_IN_YM` retired

In [ ]:
# =====================================================================
# 3 · NEW-ENTITY FEATURES, TRAILING-WINDOW DEFINITION    [OUTPUT BLOCK 2]
# =====================================================================
# "New" becomes: paid this month, and NOT paid in the previous LOOKBACK_M
# months. Stationary by construction. The first LOOKBACK_M months of the
# panel become warm-up and are NULL - derived from the data, not from a
# hard-coded list of year-months that silently goes stale the moment the
# panel start moves.
#
# Implementation is one window, not a self-join: lag the month index within
# (customer, kind, entity) and compare the gap.

feat = feat0
NEW_COLS = []

if REBUILD_NEW_ENTITY:
    pairs = spark.read.parquet(sp(PAIRS_DIR))
    disp(pd.DataFrame({"column": [c for c, _ in pairs.dtypes],
                       "dtype":  [t for _, t in pairs.dtypes]}),
         title="2a &middot; pay_pairs schema", n=40, save="v4_pairs_schema")

    k_col = PAIRS_KIND_COL   or resolve(pairs, ["kind", "entity_kind", "pair_kind", "typ", "type"],
                                        "pay_pairs kind column")
    e_col = PAIRS_ENTITY_COL or resolve(pairs, ["entity", "entity_id", "entity_key", "key",
                                                "cpty_key", "cpty_id", "unq_cpty_acct_id"],
                                        "pay_pairs entity column")
    id_col = resolve(pairs, ["cust_pwr_id"], "pay_pairs customer id", required=True)

    if k_col is None or e_col is None:
        # fall back to the two-column shape: one cpty column and one fin column
        c_alt = resolve(pairs, ["cpty_key", "cpty_id", "cpty_name", "unq_cpty_acct_id"], "cpty col")
        f_alt = resolve(pairs, ["fin_key", "fin_id", "cpty_fin_entity_name"], "fin col")
        if c_alt is None and f_alt is None:
            raise KeyError("pay_pairs shape not recognised - set PAIRS_KIND_COL / "
                           "PAIRS_ENTITY_COL from the schema printed in 2a")
        parts = []
        if c_alt: parts.append(pairs.select(F.col(id_col).alias("cust_pwr_id"), "ym",
                                            F.lit("cpty").alias("kind"),
                                            F.col(c_alt).cast("string").alias("entity")))
        if f_alt: parts.append(pairs.select(F.col(id_col).alias("cust_pwr_id"), "ym",
                                            F.lit("fin").alias("kind"),
                                            F.col(f_alt).cast("string").alias("entity")))
        pr = parts[0]
        for p in parts[1:]: pr = pr.unionByName(p)
    else:
        pr = pairs.select(F.col(id_col).alias("cust_pwr_id"), "ym",
                          F.lower(F.col(k_col).cast("string")).alias("kind"),
                          F.col(e_col).cast("string").alias("entity"))
        pr = pr.withColumn("kind", F.when(F.col("kind").rlike("fin|inst|bank"), "fin")
                                    .otherwise("cpty"))

    ymmap = cust_month.select("ym", "m_idx").distinct()
    pr = (pr.filter(F.col("entity").isNotNull())
            .join(ymmap, "ym", "inner")
            .select("cust_pwr_id", "kind", "entity", "m_idx").distinct())

    wpe = Window.partitionBy("cust_pwr_id", "kind", "entity").orderBy("m_idx")
    pr  = (pr.withColumn("prev_m", F.lag("m_idx").over(wpe))
             .withColumn("is_new", ((F.col("prev_m").isNull()) |
                                    (F.col("m_idx") - F.col("prev_m") > LOOKBACK_M)).cast("int")))

    agg = (pr.groupBy("cust_pwr_id", "m_idx")
             .agg(F.sum(F.when(F.col("kind") == "cpty", F.col("is_new"))).alias("cpty_new_out_lb"),
                  F.sum(F.when(F.col("kind") == "fin",  F.col("is_new"))).alias("fin_new_out_lb")))

    # Warm-up: before M_MIN + LOOKBACK_M the lookback is not fully observed,
    # so "new" cannot be distinguished from "first seen". NULL, not zero.
    WARM_UNTIL = M_MIN + LOOKBACK_M - 1
    agg = (agg.withColumn("cpty_new_out_lb",
                          F.when(F.col("m_idx") > WARM_UNTIL, F.col("cpty_new_out_lb")))
              .withColumn("fin_new_out_lb",
                          F.when(F.col("m_idx") > WARM_UNTIL, F.col("fin_new_out_lb"))))

    feat = (feat0.join(agg, ["cust_pwr_id", "m_idx"], "left")
                 .withColumnRenamed("cpty_new_out", "cpty_new_out_cum")
                 .withColumnRenamed("fin_new_out",  "fin_new_out_cum")
                 .withColumnRenamed("cpty_new_out_lb", "cpty_new_out")
                 .withColumnRenamed("fin_new_out_lb",  "fin_new_out")).persist(StorageLevel.DISK_ONLY)
    NEW_COLS = ["cpty_new_out", "fin_new_out"]

    # ── did it work? the same drift table on the new definition ───────
    cmp_ = (feat.groupBy("ym").agg(
               F.avg(F.col("cpty_new_out_cum").cast("double")).alias("cum_cpty"),
               F.avg(F.col("cpty_new_out").cast("double")).alias("lb_cpty"),
               F.avg(F.col("fin_new_out_cum").cast("double")).alias("cum_fin"),
               F.avg(F.col("fin_new_out").cast("double")).alias("lb_fin"))
            .orderBy("ym")).toPandas()
    disp(cmp_, title=f"2b &middot; Cumulative vs trailing-{LOOKBACK_M}m definition by calendar month. "
                     f"The lb_ columns should be flat where the cum_ columns decay. Warm-up "
                     f"months are NULL by construction",
         n=60, save="v4_new_entity_rebuilt")

    _v = cmp_.dropna(subset=["lb_cpty"])
    _stab = pd.DataFrame([
        dict(definition="cumulative (v2)", feature=c.replace("cum_", ""),
             cv=round(cmp_[c].std() / cmp_[c].mean(), 3),
             first_over_last=round(pct(cmp_[c].head(3).mean(), cmp_[c].tail(3).mean()), 2))
        for c in ("cum_cpty", "cum_fin")] + [
        dict(definition=f"trailing-{LOOKBACK_M}m (v4)", feature=c.replace("lb_", ""),
             cv=round(_v[c].std() / _v[c].mean(), 3),
             first_over_last=round(pct(_v[c].head(3).mean(), _v[c].tail(3).mean()), 2))
        for c in ("lb_cpty", "lb_fin")])
    disp(_stab, title="2c &middot; Stationarity across calendar time. A ratio near 1.0 and a low CV "
                      "is what a model can use out of time", save="v4_new_entity_stability")
    note("NEWDEF", f"Does a trailing-{LOOKBACK_M}m lookback stabilise the new-entity features?",
         "; ".join(f"{r.definition} {r.feature}: first/last {r.first_over_last}, cv {r.cv}"
                   for r in _stab.itertuples()),
         "BURN_IN_YM is retired. The warm-up is now derived from the panel start, so moving "
         "DATE_START cannot silently leave it pointing at the wrong three months.")
else:
    # Fallback if pay_pairs is unavailable: blank the warm-up by PANEL POSITION,
    # never by a hard-coded year-month.
    WARM_UNTIL = M_MIN + 2
    for c in NEW_ENTITY:
        feat = feat.withColumn(c, F.when(F.col("m_idx") > WARM_UNTIL, F.col(c)))
    warn(f"new-entity features NOT rebuilt; warm-up blanked to m_idx <= {WARM_UNTIL} "
         f"({N_MONTHS}-month panel). The calendar drift in 1b remains.")

ALL_FEATS = [f for f in (DENSE + SPARSE + ([] if DROP_MECHANIC else MECHANIC) + SHARE_LIKE)
             if f in feat.columns or f == "bal_live"]

## 3 · Labels — forward-window guard, `AB_union`, and an at-risk hazard

In [ ]:
# =====================================================================
# 4 · LABELS                                            [OUTPUT BLOCK 3]
# =====================================================================
# Three changes from v3.
#
# (a) A_REQUIRE_FWD. v3's A_full_exit fired when all_closed_run == obs_fwd.
#     At the panel edge obs_fwd is 1, so a customer whose final month shows
#     all accounts closed became an event on ONE month of confirmation while
#     an identical customer mid-panel needed three. Extending DATE_END moves
#     the artefact, it does not remove it.
#
# (b) AB_union (v3 §8.5). B fires without A on ~3k customers who are
#     currently in the CONTROL group, contaminating it in the worst possible
#     direction - they are pre-departure customers labelled as stayers.
#
# (c) The hazard denominator. events / (all customers x all evaluable
#     months) counts customers who have already left as still at risk. On a
#     31-month panel the bias was modest; on 43 months it grows, and every
#     precision figure is computed from this number.

w    = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
w12  = w.rangeBetween(-11, 0)
wfwd = w.rangeBetween(0, BAL_EXIT_HOLD - 1)
wfwA = w.rangeBetween(0, A_REQUIRE_FWD - 1)

c = (cust_month
     .withColumn("n_hist", F.count("bal_live").over(w12))
     .withColumn("med12", median_w("bal_live", w12))
     .withColumn("low", ((F.col("med12") > ZERO_TOL) &
                         (F.col("bal_live") < BAL_EXIT_FRAC * F.col("med12"))).cast("int"))
     .withColumn("low_run", F.sum("low").over(wfwd))
     .withColumn("obs_fwd", F.count("*").over(wfwd))
     .withColumn("obs_fwd_A", F.count("*").over(wfwA))
     .withColumn("all_closed_run", F.sum(COL["allcl"]).over(wfwA)))

DEFS = {
 # (a) requires A_REQUIRE_FWD observed months, all of them closed
 "A_full_exit": ((F.col(COL["allcl"]) == 1) &
                 (F.col("obs_fwd_A") == F.lit(A_REQUIRE_FWD)) &
                 (F.col("all_closed_run") == F.lit(A_REQUIRE_FWD))),
 "B_bal_exit":  ((F.col("n_hist") >= MIN_HIST_M) &
                 (F.col("low_run") == F.lit(BAL_EXIT_HOLD)) &
                 (F.col("obs_fwd") == BAL_EXIT_HOLD)),
}
for k, cond in DEFS.items():
    c = c.withColumn(k, F.when(cond, 1).otherwise(0))

# ── point-in-time attributes. v3 took F.max(segment_desc) over the whole
#    history, which is the alphabetic maximum and is not a fact about any
#    month. Entry value instead - and it has to be computed over an ORDERED
#    window, because F.first() inside a groupBy is nondeterministic and would
#    hand back a different "entry" segment on a re-run.
_ecols = [(k, COL[k]) for k in ("segment", "market", "state", "naics") if COL.get(k)]
if _ecols:
    wall = (Window.partitionBy("cust_pwr_id").orderBy("m_idx")
            .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))
    _e = cust_month.select("cust_pwr_id", "m_idx", *[cc for _, cc in _ecols])
    for k, cc in _ecols:
        _e = _e.withColumn(f"{k}_entry", F.first(F.col(cc), ignorenulls=True).over(wall))
    ATTR_ENTRY = (_e.withColumn("_rn", F.row_number().over(
                        Window.partitionBy("cust_pwr_id").orderBy("m_idx")))
                    .filter("_rn = 1")
                    .select("cust_pwr_id", *[f"{k}_entry" for k, _ in _ecols]))
else:
    ATTR_ENTRY = cust_month.select("cust_pwr_id").distinct()

lab = (c.groupBy("cust_pwr_id").agg(
          *[F.min(F.when(F.col(k) == 1, F.col("m_idx"))).alias(f"m_{k}") for k in DEFS],
          F.min("m_idx").alias("first_m"), F.max("m_idx").alias("last_m"),
          F.count("*").alias("n_months"),
          F.min(F.when(F.col(COL["nlive"]) > 0, F.col("m_idx"))).alias("first_live_m"),
          F.sum((F.col(COL["nlive"]) > 0).cast("int")).alias("n_live_months"),
          F.expr("percentile_approx(bal_live, 0.5)").alias("median_bal"),
          F.max("bal_live").alias("peak_bal"))
       .filter(f"n_months >= {MIN_HIST_M}")
       .join(ATTR_ENTRY, "cust_pwr_id", "left"))

for k in DEFS:   # qualified: real tenure inside the panel before the event
    lab = lab.withColumn(f"q_{k}", F.when(
        F.col(f"m_{k}").isNotNull() & F.col("first_live_m").isNotNull() &
        (F.col(f"m_{k}") - F.col("first_live_m") >= MIN_LIVE_BEFORE), F.col(f"m_{k}")))

# (b) union label - earliest of the two, same qualification
lab = (lab.withColumn("m_AB_union", F.least(F.col("m_A_full_exit"), F.col("m_B_bal_exit")))
          .withColumn("q_AB_union", F.least(F.col("q_A_full_exit"), F.col("q_B_bal_exit"))))

lab.write.mode("overwrite").parquet(hp("labels_customer"))
lab = spark.read.parquet(hp("labels_customer")).persist(StorageLevel.DISK_ONLY)
N_EV = lab.count()

# ── (c) at-risk denominator ───────────────────────────────────────────
# A customer-month is at risk for definition k if the customer is evaluable
# (>= MIN_HIST_M months), the month is at or after its MIN_HIST_M-th month,
# and the event has not already happened. The event month itself is at risk.
risk_base = (cust_month.select("cust_pwr_id", "m_idx")
             .join(lab.select("cust_pwr_id", "first_m",
                              *[f"q_{k}" for k in LABEL_DEFS]), "cust_pwr_id", "inner")
             .filter(F.col("m_idx") >= F.col("first_m") + MIN_HIST_M - 1))

PREV, rows = {}, []
for k in LABEL_DEFS:
    nq  = lab.filter(F.col(f"q_{k}").isNotNull()).count()
    ar  = risk_base.filter(F.col(f"q_{k}").isNull() |
                           (F.col("m_idx") <= F.col(f"q_{k}"))).count()
    old_den = N_EV * (N_MONTHS - MIN_HIST_M)
    PREV[k] = pct(nq, ar)
    rows.append(dict(definition=k,
                     n_events_raw=lab.filter(F.col(f"m_{k}").isNotNull()).count(),
                     n_events_qualified=nq,
                     share_of_customers=pct(nq, N_EV),
                     at_risk_customer_months=ar,
                     hazard_at_risk=PREV[k],
                     hazard_v3_denominator=pct(nq, old_den),
                     ratio_v4_over_v3=pct(pct(nq, ar), pct(nq, old_den))))
LAB_TBL = disp(pd.DataFrame(rows),
     title=f"3a &middot; Labels (n={N_EV:,} evaluable customers). hazard_at_risk is the base rate "
           f"an alert queue faces; the v3 column is shown so the precision tables are comparable",
     save="v4_labels")

# ── overlap and the contamination AB_union removes ────────────────────
ov = lab.agg(
        F.sum((F.col("q_A_full_exit").isNotNull() & F.col("q_B_bal_exit").isNotNull()).cast("int")).alias("A_and_B"),
        F.sum((F.col("q_A_full_exit").isNotNull() & F.col("q_B_bal_exit").isNull()).cast("int")).alias("A_only"),
        F.sum((F.col("q_A_full_exit").isNull() & F.col("q_B_bal_exit").isNotNull()).cast("int")).alias("B_only_was_control"),
        F.sum((F.col("q_AB_union").isNull()).cast("int")).alias("clean_controls"))
disp(ov, title="3b &middot; Label overlap. B_only were in v3's CONTROL group — pre-departure "
               "customers labelled as stayers", save="v4_label_overlap")

# ── does A_REQUIRE_FWD matter? count the edge events it removes ────────
_edge = (c.filter((F.col(COL["allcl"]) == 1) & (F.col("obs_fwd_A") < A_REQUIRE_FWD) &
                  (F.col("all_closed_run") == F.col("obs_fwd_A")))
          .select("cust_pwr_id").distinct().count())
kv({"customers whose only 'all closed' run sits in the final "
    f"{A_REQUIRE_FWD - 1} months": _edge,
    "…these were EVENTS in v3, are CENSORED in v4": "yes",
    "A_REQUIRE_FWD": A_REQUIRE_FWD},
   title="3c &middot; The panel-edge artefact A_REQUIRE_FWD removes", save="v4_edge_events")

note("HAZARD", "What base rate does the alert queue actually face?",
     "; ".join(f"{k}: {PREV[k]:.4%} (v3 denominator: {r['hazard_v3_denominator']:.4%})"
               for k, r in zip(LABEL_DEFS, rows)),
     "At-risk months exclude customers who have already left. Every precision figure in "
     "section 8 is computed from this, so it is not comparable to v3 without the ratio column.")

## 4 · Attributes and peer groups — the largest false-positive lever

In [ ]:
# =====================================================================
# 5 · PEER GROUPS AND PEER-RELATIVE RANKS                [OUTPUT BLOCK 4]
# =====================================================================
# A 35% fall in outbound value means nothing on its own and a great deal if
# the customer's NAICS2 x size-decile peers were flat that month. This is
# v3 §8.3 row 1, and it uses only columns already on the deposit panel.
#
# Design choices worth stating:
#  - PERCENTILE RANK within the peer cell, not a z-score. These distributions
#    are heavy-tailed; a z-score is a statement about the tail, not the node.
#  - SIZE DECILE IS FROM ENTRY, not from the current month. A declining
#    customer whose decile falls with it would be compared to a shrinking
#    peer group - the signal would normalise itself away.
#  - BACKOFF, not a single cell. A thin cell produces a rank that is mostly
#    noise, so a customer-month falls back to the next-coarsest level.
#  - NAICS keeps its three-way status. A placeholder code is not a missing
#    code and must not be pooled with one.

panel0 = (cust_month.select("cust_pwr_id", "ym", "m_idx", "bal_live",
                            *[COL[k] for k in ("nacct", "nlive", "segment", "market",
                                               "state", "naics", "balmin", "balmax")
                              if COL.get(k)])
          .join(feat.drop("ym"), ["cust_pwr_id", "m_idx"], "left"))

# ── NAICS: two digits, three-way status ───────────────────────────────
if COL.get("naics"):
    _n = F.trim(F.col(COL["naics"]).cast("string"))
    panel0 = (panel0
        .withColumn("naics_status",
            F.when(_n.isNull() | (_n == ""), "missing")
             .when(_n.rlike(r"^\*+$") | (F.upper(_n) == "UNKNOWN") | (_n == "-1"), "placeholder")
             .when(_n.rlike(r"^[0-9]{2}"), "valid").otherwise("placeholder"))
        .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring(_n, 1, 2))
                               .otherwise(F.concat(F.lit("NA_"), F.col("naics_status")))))
elif COL.get("naics_acct"):
    # NAICS lives on the account panel in some builds. One row per customer,
    # most recent non-null, then treated exactly as above.
    _na = (acct_month.select("cust_pwr_id", "m_idx", F.col(COL["naics_acct"]).alias("_naics"))
           .filter(F.col("_naics").isNotNull())
           .withColumn("_rn", F.row_number().over(
               Window.partitionBy("cust_pwr_id").orderBy(F.col("m_idx").desc())))
           .filter("_rn = 1").select("cust_pwr_id", "_naics"))
    panel0 = panel0.join(_na, "cust_pwr_id", "left")
    _n = F.trim(F.col("_naics").cast("string"))
    panel0 = (panel0
        .withColumn("naics_status",
            F.when(_n.isNull() | (_n == ""), "missing")
             .when(_n.rlike(r"^\*+$") | (F.upper(_n) == "UNKNOWN") | (_n == "-1"), "placeholder")
             .when(_n.rlike(r"^[0-9]{2}"), "valid").otherwise("placeholder"))
        .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring(_n, 1, 2))
                               .otherwise(F.concat(F.lit("NA_"), F.col("naics_status")))))
    COL["naics"] = "_naics"
else:
    panel0 = (panel0.withColumn("naics_status", F.lit("missing"))
                    .withColumn("naics2", F.lit("NA_missing")))
    warn("no NAICS column on either panel - peer groups collapse to size-decile only, "
         "which removes most of the false-positive lever peer normalisation was for")

# ── entry size decile (static per customer) ───────────────────────────
_entry = (cust_month.join(lab.select("cust_pwr_id", "first_m"), "cust_pwr_id", "inner")
          .filter(F.col("m_idx") < F.col("first_m") + 12)
          .groupBy("cust_pwr_id")
          .agg(F.expr("percentile_approx(bal_live, 0.5)").alias("size_entry")))
# ntile over a global window sorts everything into one partition. Safe HERE
# and only here: this frame is ONE ROW PER CUSTOMER (~10^5), not per
# customer-month. Never do this on the panel itself.
_entry = _entry.withColumn(
    "size_dec", F.ntile(N_SIZE_DEC).over(Window.orderBy(F.col("size_entry").asc_nulls_first())))
panel0 = panel0.join(_entry, "cust_pwr_id", "left").fillna({"size_dec": 0})

# ── tenure, from opened_dt on the account panel (no new source) ───────
if COL.get("opened"):
    _open = (acct_month.select("cust_pwr_id", COL["opened"]).dropna()
             .groupBy("cust_pwr_id").agg(F.min(F.to_date(COL["opened"])).alias("first_open_dt")))
    panel0 = (panel0.join(_open, "cust_pwr_id", "left")
              .withColumn("tenure_m",
                  F.months_between(F.to_date(F.concat_ws("-", "ym", F.lit("01"))),
                                   F.col("first_open_dt")).cast("int"))
              .withColumn("tenure_m", F.when(F.col("tenure_m") >= 0, F.col("tenure_m"))))
    TENURE = "tenure_m"
else:
    TENURE = None
    warn("no opened_dt - tenure omitted. It is normally one of the strongest static "
         "predictors of attrition hazard.")

# ── peer key with backoff ─────────────────────────────────────────────
lvl_cols, lvl_names = [], []
for i, keys in enumerate(PEER_LEVELS):
    nm = "_".join(keys) if keys else "all"
    lvl_names.append(nm)
    kexpr = F.concat_ws("|", F.lit(nm), F.col("m_idx").cast("string"),
                        *[F.coalesce(F.col(k).cast("string"), F.lit("?")) for k in keys])
    panel0 = panel0.withColumn(f"_pk{i}", kexpr)
    panel0 = panel0.withColumn(f"_pn{i}",
                               F.count("*").over(Window.partitionBy(f"_pk{i}")))
    lvl_cols.append(i)

pk = F.col(f"_pk{lvl_cols[-1]}")
for i in reversed(lvl_cols[:-1]):
    pk = F.when(F.col(f"_pn{i}") >= MIN_PEER_N, F.col(f"_pk{i}")).otherwise(pk)
panel0 = panel0.withColumn("peer_key", pk)
panel0 = panel0.withColumn("peer_level", F.split(F.col("peer_key"), r"\|").getItem(0))
panel0 = panel0.drop(*[f"_pn{i}" for i in lvl_cols], *[f"_pk{i}" for i in lvl_cols])
panel0 = panel0.persist(StorageLevel.DISK_ONLY)

disp(panel0.groupBy("peer_level").agg(F.count("*").alias("customer_months"),
        F.countDistinct("peer_key").alias("cells"),
        (F.count("*") / F.countDistinct("peer_key")).alias("mean_cell_size"))
     .orderBy(F.desc("customer_months")),
     title=f"4a &middot; Peer-group backoff — how many customer-months resolve at each level "
           f"(MIN_PEER_N={MIN_PEER_N})", save="v4_peer_levels")

if COL.get("naics"):
    disp(panel0.groupBy("naics_status").agg(F.count("*").alias("n"),
            F.countDistinct("cust_pwr_id").alias("customers"))
         .orderBy(F.desc("n")),
         title="4b &middot; NAICS three-way status. Placeholder is not missing and the two must "
               "not be pooled", save="v4_naics_status")

# ── the leakage test on segment_desc ──────────────────────────────────
# If the internal segment is restated as a customer winds down, a model that
# reads the contemporaneous value learns the restatement rather than the
# behaviour. Measure it before deciding whether to lag or freeze.
if COL.get("segment"):
    wseg = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
    seg_chg = (panel0.select("cust_pwr_id", "m_idx", F.col(COL["segment"]).alias("seg"))
               .withColumn("prev", F.lag("seg").over(wseg))
               .withColumn("chg", (F.col("prev").isNotNull() & (F.col("seg") != F.col("prev"))).cast("int"))
               .join(lab.select("cust_pwr_id", F.col("q_A_full_exit").alias("ev")), "cust_pwr_id", "left")
               .withColumn("grp", F.when(F.col("ev").isNull(), "stayer").otherwise("attriter"))
               .withColumn("rel_m", F.when(F.col("ev").isNotNull(), F.col("m_idx") - F.col("ev"))))
    st = (seg_chg.filter(F.col("grp") == "stayer").agg(F.avg("chg").alias("rate")).collect()[0][0])
    at = (seg_chg.filter((F.col("grp") == "attriter") & F.col("rel_m").between(-12, -1))
          .groupBy("rel_m").agg(F.avg("chg").alias("rate"), F.count("*").alias("n"))
          .orderBy("rel_m")).toPandas()
    at["stayer_baseline"] = st
    at["lift"] = (at["rate"] / st).round(2)
    disp(at, title="4c &middot; Is segment_desc restated in the run-up to an exit? A lift well "
                   "above 1 near rel_m 0 means the contemporaneous value leaks",
         n=20, save="v4_segment_leakage")
    _lift = float(at.loc[at.rel_m >= -3, "lift"].max()) if len(at) else float("nan")
    note("SEGLEAK", "Does segment_desc leak the outcome?",
         ("YES - restated near the event; use the entry value or lag by ATTR_LAG_M"
          if _lift > 1.5 else f"No material restatement (max lift {_lift:.2f})"),
         "Internal hierarchical segments are usually assigned from revenue or balance tiers "
         "and refreshed periodically. The model matrix in section 10 lags every static "
         f"attribute by ATTR_LAG_M={ATTR_LAG_M} regardless.")

## 5 · Trend block and peer-relative ranks

In [ ]:
# =====================================================================
# 6 · TREND FEATURES AND PEER RANKS                      [OUTPUT BLOCK 5]
# =====================================================================
# The event curves in v3 §6.2 are entirely about SHAPE, and the 21-feature
# set encodes only LEVEL. A 20% fall that has been sliding all year is a
# different customer from one that fell last month, and nothing in the
# feature set could tell them apart.
#
# Trailing MEANS, not medians - a collect_list median over a 12-month window
# on seven features is an expensive way to buy very little here.
# rangeBetween on m_idx, never rowsBetween: a customer with a gap month must
# not have its window silently shift.

wc  = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
TREND_COLS = []
for f in [x for x in TREND_FEATS if x in panel0.columns]:
    for k in TREND_WINDOWS:
        wk = wc.rangeBetween(-k, -1)
        panel0 = panel0.withColumn(f"{f}_r{k}",
            F.when(F.abs(F.avg(F.col(f).cast("double")).over(wk)) > 1e-9,
                   F.col(f).cast("double") / F.avg(F.col(f).cast("double")).over(wk)))
        TREND_COLS.append(f"{f}_r{k}")
    # consecutive-decline count over the trailing 6 (a run-length proxy)
    panel0 = panel0.withColumn(f"_d_{f}",
        (F.col(f).cast("double") < F.lag(F.col(f).cast("double")).over(wc)).cast("int"))
    for k in TREND_WINDOWS:
        panel0 = panel0.withColumn(f"{f}_dec{k}", F.sum(f"_d_{f}").over(wc.rangeBetween(-(k - 1), 0)))
        TREND_COLS.append(f"{f}_dec{k}")
    # months since the trailing-PEAK_WINDOW peak
    wp = wc.rangeBetween(-(PEAK_WINDOW - 1), 0)
    panel0 = (panel0
        .withColumn(f"_pk_{f}", F.max(F.col(f).cast("double")).over(wp))
        .withColumn(f"_at_{f}", F.when(F.col(f).cast("double") >= F.col(f"_pk_{f}") * 0.999,
                                       F.col("m_idx")))
        .withColumn(f"{f}_since_peak", F.col("m_idx") - F.max(f"_at_{f}").over(wp)))
    TREND_COLS.append(f"{f}_since_peak")
    if YOY_OK:
        panel0 = panel0.withColumn(f"{f}_yoy",
            F.when(F.abs(F.avg(F.col(f).cast("double")).over(wc.rangeBetween(-12, -12))) > 1e-9,
                   F.col(f).cast("double") /
                   F.avg(F.col(f).cast("double")).over(wc.rangeBetween(-12, -12))))
        TREND_COLS.append(f"{f}_yoy")
    panel0 = panel0.drop(f"_d_{f}", f"_pk_{f}", f"_at_{f}")

# balance behaviour, if bal_min / bal_max are on the panel (v3 §8.3 row 8)
BAL_COLS = []
if COL.get("balmin") and COL.get("balmax"):
    panel0 = (panel0
        .withColumn("bal_swing", F.when(F.col("bal_live") > ZERO_TOL,
                    (F.col(COL["balmax"]) - F.col(COL["balmin"])) / F.col("bal_live")))
        .withColumn("bal_at_zero", (F.col(COL["balmin"]) <= ZERO_TOL).cast("int")))
    BAL_COLS = ["bal_swing", "bal_at_zero"]

# account-level early warning (v3 §8.3 row 7): 33,428 closures belong to
# customers who stayed, and closing one of five precedes closing all five.
if COL.get("nacct"):
    panel0 = (panel0
        .withColumn("_nl", F.col(COL["nlive"]).cast("double"))
        .withColumn("acct_lost_3m", F.greatest(
            F.lit(0), F.max("_nl").over(wc.rangeBetween(-3, 0)) - F.col("_nl")))
        .withColumn("acct_share_live", F.when(F.col(COL["nacct"]) > 0,
                    F.col("_nl") / F.col(COL["nacct"]).cast("double")))
        .drop("_nl"))
    BAL_COLS += ["acct_lost_3m", "acct_share_live"]

# ── lagged static attributes, on the FULL panel and by rangeBetween.
# Lagging inside the model matrix would use rowsBetween over a panel already
# filtered to at-risk months, so "6 months ago" would silently mean "6 rows
# ago" and would differ per customer.
LAG_COLS = []
for c in [COL.get("segment"), COL.get("market"), COL.get("state")]:
    if c:
        panel0 = panel0.withColumn(f"{c}_lag",
            F.max(F.col(c).cast("string")).over(wc.rangeBetween(-ATTR_LAG_M, -ATTR_LAG_M)))
        LAG_COLS.append(f"{c}_lag")
if TENURE:
    panel0 = panel0.withColumn("tenure_m_lag",
        F.max(F.col(TENURE)).over(wc.rangeBetween(-ATTR_LAG_M, -ATTR_LAG_M)))
    LAG_COLS.append("tenure_m_lag")

# ── materialise. The trend block alone stacks ~35 window operations on top
# of the peer-key windows, and the rank block below adds another ~30. Spark
# will happily build a single plan out of all of it and then die in the
# optimiser. Write and read back to cut the lineage; this is the same lesson
# as "one job per month" in the v2 payment build.
if MATERIALISE_PANEL:
    panel0.write.mode("overwrite").parquet(hp("panel_features"))
    panel = spark.read.parquet(hp("panel_features")).persist(StorageLevel.DISK_ONLY)
else:
    panel = panel0.persist(StorageLevel.DISK_ONLY)
print(f"trend columns: {len(TREND_COLS)}   balance/account columns: {len(BAL_COLS)}   "
      f"lagged static: {len(LAG_COLS)}   rows: {panel.count():,}")

# ── peer percentile ranks ─────────────────────────────────────────────
# rank / n_nonnull rather than percent_rank(), because percent_rank divides
# by the partition size INCLUDING nulls and would compress every rank in a
# sparse cell toward zero.
RANK_FEATS = ([f for f in DENSE if f in panel.columns] +
              [f for f in ("cpty_new_out", "fin_new_out", "share_out_selfpay",
                           "share_out_internal", "share_out_origpnc") if f in panel.columns] +
              [c for c in TREND_COLS if c.endswith(("_r3", "_r6", "_yoy"))] +
              [c for c in BAL_COLS if c in ("bal_swing", "acct_share_live")])
RANK_FEATS = list(dict.fromkeys(RANK_FEATS))

PR_COLS = []
for f in RANK_FEATS:
    wp = Window.partitionBy("peer_key")
    wo = Window.partitionBy("peer_key").orderBy(F.col(f).cast("double").asc_nulls_last())
    nn = F.sum(F.col(f).isNotNull().cast("int")).over(wp)
    panel = panel.withColumn(f"{f}_pr",
        F.when(F.col(f).isNotNull() & (nn > 1),
               (F.rank().over(wo) - 1) / (nn - 1)))
    PR_COLS.append(f"{f}_pr")

if MATERIALISE_PANEL:
    panel.write.mode("overwrite").parquet(hp("panel_ranked"))
    panel = spark.read.parquet(hp("panel_ranked")).persist(StorageLevel.DISK_ONLY)
else:
    panel = panel.persist(StorageLevel.DISK_ONLY)

_cov = panel.agg(*[F.avg(F.col(c).isNotNull().cast("double")).alias(c) for c in PR_COLS[:40]])
disp(_cov.toPandas().T.reset_index().rename(columns={"index": "peer_rank_column", 0: "coverage"}),
     title=f"5a &middot; Peer-rank coverage ({len(PR_COLS)} rank columns over {len(RANK_FEATS)} "
           f"features). A low value is a sparse feature, not a broken rank",
     n=60, save="v4_peer_rank_coverage")

note("FEATBLOCK", "How large is the feature set now?",
     f"{len(ALL_FEATS)} raw + {len(TREND_COLS)} trend + {len(BAL_COLS)} balance/account + "
     f"{len(PR_COLS)} peer ranks",
     "All derived from columns already on the panel. No new extract.")

## 6 · Event study — two horizon configurations, proper control draw

In [ ]:
# =====================================================================
# 7 · EVENT STUDY                                       [OUTPUT BLOCK 6]
# =====================================================================
# Two fixes and one addition.
#
# (a) The control draw. v3 built the pseudo-event distribution from
#     attr.select("event_m").limit(2000) - whatever partitions returned
#     first, then strided. Replaced with a draw from the empirical
#     event-month distribution.
#
# (b) Controls are drawn from customers clean under AB_union, so the ~3k
#     B-only customers stop sitting in the control group.
#
# (c) The study is run at every configuration in RUN_CONFIGS. H18 exists to
#     find out whether the v3 answer was the signal or the edge of the
#     search window.

SPARSE_SET = set(SPARSE) | set(NEW_COLS)

def _event_month_array(defn, n_slots=1000):
    """Empirical event-month distribution, expanded to a fixed-length array
    so a control can be assigned by a single uniform draw."""
    d = (lab.filter(F.col(f"q_{defn}").isNotNull())
            .groupBy(F.col(f"q_{defn}").alias("em")).count().orderBy("em")).toPandas()
    if d.empty:
        return [0]
    p = (d["count"] / d["count"].sum()).values
    slots = np.maximum(1, np.round(p * n_slots).astype(int))
    return list(np.repeat(d["em"].values.astype(int), slots))

def cohorts_for(defn, EVENT_PRE):
    attr = (lab.filter(F.col(f"q_{defn}").isNotNull())
            .select("cust_pwr_id", F.col(f"q_{defn}").alias("event_m"),
                    F.col("first_live_m").alias("first_m"), "last_m", "peak_bal")
            .withColumn("cohort", F.lit("attriter")))
    draw = _event_month_array(defn)
    arr  = F.array(*[F.lit(int(x)) for x in draw])
    ctrl = (lab.filter(F.col("q_AB_union").isNull())          # (b)
            .withColumn("_u", F.floor(F.rand(SEED) * F.lit(len(draw))) + 1)   # (a)
            .withColumn("event_m", F.element_at(arr, F.col("_u").cast("int")))
            .withColumn("first_m", F.coalesce("first_live_m", "first_m"))
            .select("cust_pwr_id", "event_m", "first_m", "last_m", "peak_bal")
            .withColumn("cohort", F.lit("stayer")))
    return (attr.unionByName(ctrl)
            .filter((F.col("first_m") <= F.col("event_m") - EVENT_PRE) &
                    (F.col("last_m") >= F.col("event_m"))))

_stack = ", ".join([f"'{f}', CAST({f} AS DOUBLE)" for f in ALL_FEATS])

def normalise(defn, cfg):
    EVENT_PRE  = cfg["EVENT_PRE"]
    BASE_WINDOW = cfg["BASE_WINDOW"]
    co = cohorts_for(defn, EVENT_PRE)
    es = (panel.join(co, "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
          .filter(F.col("rel_m").between(-EVENT_PRE, EVENT_POST)))
    lg = es.select("cust_pwr_id", "cohort", "rel_m", "peak_bal",
                   F.expr(f"stack({len(ALL_FEATS)}, {_stack}) as (feature, value)"))
    if BASE_MODE == "seasonal":
        anchors = [a for a in BASE_ANCHORS if a >= -EVENT_PRE]
        bfilter = F.col("rel_m").isin(*anchors)
    else:
        bfilter = F.col("rel_m").between(*BASE_WINDOW)
    base = (lg.filter(bfilter).groupBy("cust_pwr_id", "feature")
              .agg(F.avg("value").alias("base")))
    nm = (lg.join(base, ["cust_pwr_id", "feature"], "left")
          .withColumn("idx", F.when(F.abs(F.col("base")) > 1e-9, F.col("value") / F.col("base"))))
    return co, nm.persist(StorageLevel.DISK_ONLY)

CURVES, NORMS, COH = {}, {}, {}
for cname in RUN_CONFIGS:
    cfg = CONFIGS[cname]
    for d in STUDY_DEFS:
        co, nm = normalise(d, cfg)
        COH[(cname, d)], NORMS[(cname, d)] = co, nm
        cur = (nm.groupBy("feature", "cohort", "rel_m").agg(
                   F.count("*").alias("n"),
                   F.sum(F.col("idx").isNotNull().cast("int")).alias("n_idx"),
                   F.expr("percentile_approx(idx, 0.5)").alias("med_idx"),
                   F.avg(F.when(F.col("value").isNotNull(),
                                (F.col("value") > 0).cast("double"))).alias("rate_any"),
                   F.avg("value").alias("mean_value"))).toPandas()
        cur.loc[cur.n < MIN_CELL_N, ["med_idx", "rate_any", "mean_value"]] = np.nan
        CURVES[(cname, d)] = cur
        cur.to_csv(OUT_DIR / f"v4_curves_{cname}_{d}.csv", index=False)

sizes = pd.DataFrame([
    dict(config=cn, definition=d,
         EVENT_PRE=CONFIGS[cn]["EVENT_PRE"], SEARCH_FROM=CONFIGS[cn]["SEARCH_FROM"],
         attriters=COH[(cn, d)].filter("cohort='attriter'").count(),
         stayers=COH[(cn, d)].filter("cohort='stayer'").count())
    for cn in RUN_CONFIGS for d in STUDY_DEFS])
disp(sizes, title="6a &middot; Cohort sizes. H18 buys nine more months of pre-window and pays for "
                  "it in cohort size — that trade is the whole point of running both",
     save="v4_cohorts")

HEAD = [f for f in ["bal_live", "amt_out", "amt_in", "n_out", "cpty_out_n"] if f in ALL_FEATS]
def curve_table(key, feats, col):
    cur = CURVES[key]
    return (cur[cur.feature.isin(feats)]
            .pivot_table(index="rel_m", columns=["feature", "cohort"], values=col)
            .reindex(columns=pd.MultiIndex.from_product([feats, ["attriter", "stayer"]]))
            .round(3).reset_index())

for cn in RUN_CONFIGS:
    disp(curve_table((cn, "A_full_exit"), HEAD, "med_idx"),
         title=f"6b &middot; {cn} · A_full_exit — median ratio to own baseline",
         n=CONFIGS[cn]["EVENT_PRE"] + EVENT_POST + 1, save=f"v4_curve_{cn}_A_dense")

RATE_HEAD = [f for f in ("cpty_new_out", "fin_new_out", "share_out_selfpay") if f in ALL_FEATS]
if RATE_HEAD:
    disp(curve_table((PRIMARY_CFG, "A_full_exit"), RATE_HEAD, "rate_any"),
         title=f"6c &middot; {PRIMARY_CFG} · A_full_exit — RATE of any non-zero, on the REBUILT "
               f"trailing-window new-entity definition",
         n=CONFIGS[PRIMARY_CFG]["EVENT_PRE"] + EVENT_POST + 1, save="v4_curve_A_rate")

## 7 · Lead detector — and the censoring test

In [ ]:
# =====================================================================
# 8 · LEAD DETECTOR + CENSORING TEST                     [OUTPUT BLOCK 7]
# =====================================================================
# v3's headline was cpty_new_out separating at rel_m -9, which IS the first
# month the search looked at. A separation that lands on SEARCH_FROM is not
# a measurement of the lead - it is a lower bound, and the notebook has to
# say so rather than printing a number.

def separations(key):
    cname, defn = key
    SEARCH_FROM = CONFIGS[cname]["SEARCH_FROM"]
    cur, rows = CURVES[key], []
    for f, g in cur.groupby("feature"):
        if f in SPARSE_SET:
            col, thr, kind = "rate_any", SEP_RATE, "rate"
        elif f in MECHANIC or f in SHARE_LIKE:
            col, thr, kind = "med_idx", SEP_SHARE, "share"
        else:
            col, thr, kind = "med_idx", SEP_LEVEL, "level"
        w = (g.pivot_table(index="rel_m", columns="cohort", values=col)
               .reindex(columns=["attriter", "stayer"]).dropna().sort_index())
        if w.empty:
            continue
        gap = (w.attriter - w.stayer).abs()
        search = gap[gap.index >= SEARCH_FROM]
        sep, run = None, 0
        for rm, v in search.items():
            run = run + 1 if v > thr else 0
            if run >= HOLD:
                sep = rm - HOLD + 1
                break
        censored = (sep is not None) and (sep <= SEARCH_FROM + HOLD - 1)
        rows.append(dict(feature=f, kind=kind, threshold=thr,
                         first_separation_rel_m=sep,
                         lead_months=(None if sep is None else -sep),
                         CENSORED=censored,
                         gap_at_search_start=round(gap.get(SEARCH_FROM, np.nan), 3),
                         max_gap_in_search=round(search.max(), 3) if len(search) else np.nan,
                         mechanical=f in MECHANIC))
    return pd.DataFrame(rows).sort_values(
        ["first_separation_rel_m", "max_gap_in_search"], ascending=[True, False],
        na_position="last")

LEADS = {}
for cn in RUN_CONFIGS:
    for d in STUDY_DEFS:
        t = separations((cn, d))
        LEADS[(cn, d)] = t
        disp(t, title=f"7a &middot; {cn} · {d} — first sustained separation, search from rel_m "
                      f"{CONFIGS[cn]['SEARCH_FROM']}. CENSORED = it separated at the window edge, "
                      f"so the true lead is at least this and possibly more",
             n=40, save=f"v4_lead_{cn}_{d}")

def headline(cn, d="A_full_exit"):
    t = LEADS[(cn, d)]
    b = t[t.feature == "bal_live"]
    p = t[(t.feature != "bal_live") & t.first_separation_rel_m.notna() & (~t.mechanical)]
    bal = None if b.empty else b.iloc[0].first_separation_rel_m
    pay = None if p.empty else p.iloc[0]
    return dict(config=cn, definition=d,
                bal_live_sep=bal,
                earliest_payment_feature=None if pay is None else pay.feature,
                payment_sep=None if pay is None else pay.first_separation_rel_m,
                payment_censored=None if pay is None else bool(pay.CENSORED),
                lead_months=None if (pay is None or bal is None)
                            else int(bal - pay.first_separation_rel_m),
                search_from=CONFIGS[cn]["SEARCH_FROM"])

HL = pd.DataFrame([headline(cn, d) for cn in RUN_CONFIGS for d in STUDY_DEFS])
disp(HL, title="7b &middot; Headline across configurations. If the lead GROWS from H12 to H18, "
               "the v3 number was censored and the real lead is the H18 one",
     save="v4_headline")

_h12 = HL[(HL.config == "H12") & (HL.definition == "A_full_exit")]
_h18 = HL[(HL.config == "H18") & (HL.definition == "A_full_exit")]
_verdict = "inconclusive"
if not _h12.empty and not _h18.empty:
    a, b = _h12.iloc[0], _h18.iloc[0]
    if a.payment_censored and b.lead_months is not None and a.lead_months is not None:
        _verdict = ("CENSORED in H12 - H18 lead is "
                    f"{b.lead_months}m vs H12 {a.lead_months}m" if b.lead_months > a.lead_months
                    else f"H12 was censored but H18 agrees at {b.lead_months}m")
    elif a.lead_months is not None:
        _verdict = f"not censored - {a.lead_months}m stands"
note("LEAD", "Does payment behaviour lead the deposit balance, and by how much?",
     _verdict,
     "v3 reported 4 months from cpty_new_out at rel_m -9, which was SEARCH_FROM. H18 pushes "
     "the search back to -15 on a smaller cohort; the comparison is what separates a measured "
     "lead from a lower bound.")

## 8 · Operating points — all features, uncontaminated months only

In [ ]:
# =====================================================================
# 9 · OPERATING POINTS                                   [OUTPUT BLOCK 8]
# =====================================================================
# Three v3 defects fixed.
#   7.1  cpty_new_out and fin_new_out are IN OP_FEATURES. They separate
#        first and had no precision or recall attached to them at all.
#   7.2  rel_m is restricted to <= OP_MAX_REL_M. -12..-10 is the baseline
#        window where the normalised value is ~1.0 by construction.
#   7.3  the fallback path is labelled as a fallback in the output, not in
#        a comment.
# Plus: sparse counts get a "fell to exactly zero" cut, which is the only
# threshold that means anything for a feature whose median is zero.

OP_FEATURES = [f for f in (DENSE + NEW_COLS) if f in ALL_FEATS]

def op_points(key):
    cname, defn = key
    EVENT_PRE = CONFIGS[cname]["EVENT_PRE"]
    nm = NORMS[key]
    agg = [F.count("*").alias("n"),
           F.sum(F.col("idx").isNotNull().cast("int")).alias("n_idx"),
           F.sum(F.col("value").isNotNull().cast("int")).alias("n_val"),
           F.sum((F.col("value") <= 0).cast("int")).alias("n_zero")]
    agg += [F.sum((F.col("idx") < t).cast("int")).alias(f"lt{i}")
            for i, t in enumerate(OP_THRESH)]
    raw = (nm.filter(F.col("feature").isin(*OP_FEATURES) &
                     F.col("rel_m").between(-EVENT_PRE, OP_MAX_REL_M))
             .groupBy("feature", "rel_m", "cohort").agg(*agg)).toPandas()

    p, out = PREV[defn], []
    for (f, rm), g in raw.groupby(["feature", "rel_m"]):
        a = g[g.cohort == "attriter"]; s = g[g.cohort == "stayer"]
        if a.empty or s.empty:
            continue
        a, s = a.iloc[0], s.iloc[0]
        cuts = [(f"idx<{t}", (a[f"lt{i}"], a.n_idx), (s[f"lt{i}"], s.n_idx))
                for i, t in enumerate(OP_THRESH)]
        if OP_ZERO_RULE:
            cuts.append(("value==0", (a.n_zero, a.n_val), (s.n_zero, s.n_val)))
        for label, (na, da), (ns, ds) in cuts:
            if da < MIN_CELL_N or ds < MIN_CELL_N:
                continue
            rec, fpr = na / da, ns / ds
            den = p * rec + (1 - p) * fpr
            prec = (p * rec / den) if den > 0 else np.nan
            out.append(dict(feature=f, rel_m=int(rm), cut=label, recall=rec, fpr=fpr,
                            precision=prec,
                            alerts_per_true_positive=(1 / prec if prec and prec > 0 else np.nan),
                            n_attriter=int(da), n_stayer=int(ds)))
    return pd.DataFrame(out)

OP = {}
for d in STUDY_DEFS:
    OP[d] = op_points((PRIMARY_CFG, d))
    OP[d].to_csv(OUT_DIR / f"v4_operating_points_{PRIMARY_CFG}_{d}.csv", index=False)

def best(df):
    ok = df[(df.precision >= TARGET_PREC) & (df.recall >= MIN_RECALL)]
    if ok.empty:
        idx = df.groupby(["feature", "rel_m"]).precision.idxmax().dropna()
        return df.loc[idx].assign(meets_target=False), False
    idx = ok.groupby(["feature", "rel_m"]).recall.idxmax()
    return ok.loc[idx].assign(meets_target=True), True

b, MET = best(OP["A_full_exit"])
_cap = (f"recall at precision &ge; {TARGET_PREC:.0%}" if MET else
        "<b>FALLBACK — nothing reaches the precision target, so this is recall at the "
        "best-precision cut</b>")
disp(b.pivot_table(index="feature", columns="rel_m", values="recall").round(3).reset_index(),
     title=f"8a &middot; A_full_exit · {_cap}", n=25, save="v4_op_recall")
disp(b.pivot_table(index="feature", columns="rel_m", values="precision").round(3).reset_index(),
     title="8b &middot; Best achievable precision by month before exit "
           f"(rel_m &le; {OP_MAX_REL_M}; the baseline window is excluded)",
     n=25, save="v4_op_precision")

brief = []
for f, g in OP["A_full_exit"].groupby("feature"):
    ok = g[(g.precision >= TARGET_PREC) & (g.recall >= MIN_RECALL)]
    bp = g.loc[g.precision.idxmax()] if len(g) else None
    e  = None if ok.empty else ok.loc[ok.rel_m.idxmin()]
    brief.append(dict(feature=f,
                      earliest_usable_rel_m=None if e is None else int(e.rel_m),
                      cut=None if e is None else e.cut,
                      recall=None if e is None else round(e.recall, 3),
                      precision=None if e is None else round(e.precision, 3),
                      best_precision_anywhere=None if bp is None else round(bp.precision, 3),
                      best_cut=None if bp is None else bp.cut,
                      lift_over_base=None if bp is None else round(bp.precision / PREV["A_full_exit"], 1)))
brief = pd.DataFrame(brief).sort_values("best_precision_anywhere", ascending=False)
disp(brief, title=f"8c &middot; Earliest month each feature supports a usable queue "
                  f"(precision &ge; {TARGET_PREC:.0%}, recall &ge; {MIN_RECALL:.0%}) — now "
                  f"including the two earliest signals, which v3 never scored",
     n=25, save="v4_op_brief")

_ne = brief[brief.feature.isin(NEW_COLS)]
note("OPNEW", "What are the earliest-separating features actually worth?",
     ("; ".join(f"{r.feature}: best precision {r.best_precision_anywhere} "
                f"({r.lift_over_base}x base) at cut {r.best_cut}" for r in _ne.itertuples())
      or "new-entity features not scored - check the rebuild in section 3"),
     "v3 §7.1 flagged this as the first thing to fix: cpty_new_out and fin_new_out separated "
     "at rel_m -9 and were left out of OP_FEATURES, so the two signals that fire earliest had "
     "no precision attached to them.")

## 9 · Is HHI anything beyond counterparty count? (re-proof on the longer panel)

In [ ]:
# =====================================================================
# 10 · MECHANICAL FEATURES, RE-PROVED                    [OUTPUT BLOCK 9]
# =====================================================================
# v3 held the counterparty count fixed and the attriter/stayer HHI gap
# vanished, so the four concentration features were dropped. Re-prove it on
# the longer panel rather than inheriting the conclusion - it is one cell,
# and a dropped feature that turns out to be real is an expensive mistake.

if all(c in cust_month.columns or c in feat.columns for c in ("cpty_out_hhi", "cpty_out_n")):
    es = (panel.join(COH[(PRIMARY_CFG, "A_full_exit")], "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
          .filter(F.col("rel_m").between(-CONFIGS[PRIMARY_CFG]["EVENT_PRE"], 0))
          .filter(F.col("cpty_out_n").isNotNull() & F.col("cpty_out_hhi").isNotNull())
          .withColumn("n_bucket",
                      F.when(F.col("cpty_out_n") <= 2, "1-2")
                       .when(F.col("cpty_out_n") <= 5, "3-5")
                       .when(F.col("cpty_out_n") <= 15, "6-15")
                       .when(F.col("cpty_out_n") <= 50, "16-50").otherwise("51+")))
    cond = (es.groupBy("n_bucket", "cohort")
              .agg(F.count("*").alias("n"),
                   F.expr("percentile_approx(cpty_out_hhi, 0.5)").alias("median_hhi"))
              .filter(F.col("n") >= MIN_CELL_N)).toPandas()
    cp = (cond.pivot_table(index="n_bucket", columns="cohort", values="median_hhi")
            .reindex(["1-2", "3-5", "6-15", "16-50", "51+"]))
    cp["gap"] = (cp.get("attriter", np.nan) - cp.get("stayer", np.nan)).abs().round(4)
    disp(cp.reset_index(), title="9a &middot; Within a fixed counterparty-count bucket, does HHI "
                                 "still separate? A small gap confirms the v3 drop",
         save="v4_hhi_conditional")
    _mg = float(cp["gap"].max())
    note("HHI", "Is cpty_out_hhi informative beyond cpty_out_n, on the longer panel?",
         "NO - drop confirmed" if _mg < 0.05 else f"RECONSIDER - max within-bucket gap {_mg:.3f}",
         "v3 measured a max gap of 0.008. If the longer panel disagrees, DROP_MECHANIC must "
         "be revisited before the model is built.")
else:
    print("concentration features not on the panel - nothing to re-prove")

## 10 · The deliverable — a ranked worklist, not a threshold

In [ ]:
# =====================================================================
# 11 · RANKED WORKLIST — PRECISION@K                    [OUTPUT BLOCK 10]
# =====================================================================
# v3 §8.1. Every number in section 8 measures a BINARY CUT, which throws
# away magnitude and forces the precision/recall trade-off that made the
# frontier's top-right corner empty. A score that ranks the whole book lets
# the team take the top k names - whatever they can actually work - and
# precision on that slice is far above any global threshold, because the top
# of the list is extreme on several measures at once.
#
# The score here is deliberately NOT a trained model. It is the mean of the
# risk-oriented peer percentile ranks: zero parameters, no training set, no
# leakage, auditable line by line. It is the number a gradient-boosted
# hazard model has to beat, and if it does not beat it by much then the
# model was not the bottleneck.

SCORE_PARTS = []
for f in RANK_FEATS:
    base = f
    for suf in ("_r3", "_r6", "_yoy", "_dec3", "_dec6", "_since_peak"):
        if f.endswith(suf):
            base = f[: -len(suf)]
            break
    d = RISK_DIRECTION.get(base, -1)
    if d == 0 or f"{f}_pr" not in panel.columns:
        continue          # an ambiguous direction adds a constant, not evidence
    SCORE_PARTS.append(F.lit(1.0) - F.col(f"{f}_pr") if d < 0 else F.col(f"{f}_pr"))
print(f"score built from {len(SCORE_PARTS)} peer-rank components")

_n = F.lit(0)
_s = F.lit(0.0)
for p in SCORE_PARTS:
    _n = _n + F.when(p.isNotNull(), 1).otherwise(0)
    _s = _s + F.coalesce(p, F.lit(0.0))
scored = (panel.withColumn("_np", _n).withColumn("_sp", _s)
          .withColumn("risk_score", F.when(F.col("_np") >= 3, F.col("_sp") / F.col("_np")))
          .drop("_np", "_sp"))

# ── at-risk rows and forward labels ───────────────────────────────────
DEFN = WORKLIST_DEF
wl = (scored.join(lab.select("cust_pwr_id", "first_m", "last_m",
                             F.col(f"q_{DEFN}").alias("event_m")), "cust_pwr_id", "inner")
      .filter((F.col("m_idx") >= F.col("first_m") + MIN_HIST_M - 1) &
              (F.col("event_m").isNull() | (F.col("m_idx") < F.col("event_m")))))
for h in HORIZONS:
    wl = wl.withColumn(f"y{h}", F.when(F.col("event_m").isNotNull() &
                                       (F.col("event_m") - F.col("m_idx") <= h), 1).otherwise(0))
# the last h months cannot be scored at horizon h - the outcome is unobserved
wl = wl.withColumn("months_to_end", F.lit(M_MAX) - F.col("m_idx"))
wl = wl.filter(F.col("risk_score").isNotNull()).persist(StorageLevel.DISK_ONLY)

# ── precision@k, month by month, then averaged ────────────────────────
wrank = Window.partitionBy("m_idx").orderBy(F.col("risk_score").desc_nulls_last())
wl = wl.withColumn("rk", F.row_number().over(wrank))

rows = []
for h in HORIZONS:
    e = wl.filter(F.col("months_to_end") >= h)
    tot = (e.groupBy("m_idx").agg(F.sum(f"y{h}").alias("pos"),
                                  F.sum(F.when(F.col(f"y{h}") == 1, F.col("bal_live"))).alias("pos_bal"),
                                  F.count("*").alias("at_risk")))
    for k in TOPK:
        top = (e.filter(F.col("rk") <= k).groupBy("m_idx")
                 .agg(F.sum(f"y{h}").alias("tp"),
                      F.sum(F.when(F.col(f"y{h}") == 1, F.col("bal_live"))).alias("tp_bal"),
                      F.count("*").alias("k_actual")))
        j = (top.join(tot, "m_idx")
               .agg(F.avg(F.col("tp") / F.col("k_actual")).alias("precision_at_k"),
                    F.avg(F.col("tp") / F.col("pos")).alias("recall_at_k"),
                    F.avg(F.col("tp_bal") / F.col("pos_bal")).alias("dollar_recall_at_k"),
                    F.avg("pos").alias("mean_positives_per_month"),
                    F.avg("at_risk").alias("mean_at_risk"),
                    F.count("*").alias("months")).collect()[0].asDict())
        j.update(horizon=h, k=k)
        j["lift_over_base"] = pct(j["precision_at_k"], pct(j["mean_positives_per_month"],
                                                           j["mean_at_risk"]))
        j["true_positives_per_month"] = j["precision_at_k"] * k
        rows.append(j)

PK = pd.DataFrame(rows)[["horizon", "k", "months", "mean_at_risk", "mean_positives_per_month",
                         "precision_at_k", "true_positives_per_month", "recall_at_k",
                         "dollar_recall_at_k", "lift_over_base"]].round(4)
disp(PK, title=f"10a &middot; precision@k on the untrained rank score ({DEFN}). "
               f"true_positives_per_month is the number the sales lead will ask for",
     n=40, save="v4_precision_at_k")

# ── out-of-time read: the same table by calendar half ─────────────────
wl = wl.withColumn("period", F.concat(F.substring("ym", 1, 4), F.lit("H"),
                                      F.when(F.substring("ym", 6, 2).cast("int") <= 6, "1").otherwise("2")))
h_ = 6 if 6 in HORIZONS else HORIZONS[-1]
k_ = 250 if 250 in TOPK else TOPK[len(TOPK) // 2]
e = wl.filter(F.col("months_to_end") >= h_)
tot = e.groupBy("m_idx", "period").agg(F.sum(f"y{h_}").alias("pos"), F.count("*").alias("at_risk"))
top = (e.filter(F.col("rk") <= k_).groupBy("m_idx", "period")
         .agg(F.sum(f"y{h_}").alias("tp"), F.count("*").alias("k_actual")))
oot = (top.join(tot, ["m_idx", "period"]).groupBy("period")
         .agg(F.count("*").alias("months"),
              F.avg(F.col("tp") / F.col("k_actual")).alias("precision_at_k"),
              F.avg(F.col("tp") / F.col("pos")).alias("recall_at_k"),
              F.avg("pos").alias("positives_per_month"))
         .orderBy("period"))
disp(oot, title=f"10b &middot; Stability across calendar periods, horizon {h_}m, k={k_}. v3 §7.5 — "
                f"nothing had ever been checked out of time. A score that swings here will not "
                f"survive a rolling-origin fold",
     n=20, save="v4_precision_at_k_by_period")

# ── the honest comparison: score vs the best single threshold ─────────
_bestthr = brief.best_precision_anywhere.max() if len(brief) else np.nan
_p6 = PK[(PK.horizon == h_) & (PK.k == k_)]
note("WORKLIST", f"What does a ranked worklist of {k_} names at {h_} months deliver?",
     ("" if _p6.empty else
      f"precision {_p6.iloc[0].precision_at_k:.1%}, "
      f"{_p6.iloc[0].true_positives_per_month:.0f} genuine departures/month, "
      f"recall {_p6.iloc[0].recall_at_k:.1%}, dollar-recall {_p6.iloc[0].dollar_recall_at_k:.1%}, "
      f"{_p6.iloc[0].lift_over_base:.0f}x base"),
     f"Best single-feature threshold anywhere in section 8: {_bestthr}. This score has zero "
     f"trained parameters - it is the floor a hazard model has to clear, not the ceiling. "
     f"Target from the v3 handoff §8.7 was ~250 names containing 70-80 departures.")

## 11 · Model matrix — the handoff to v5

In [ ]:
# =====================================================================
# 12 · MODEL MATRIX                                     [OUTPUT BLOCK 11]
# =====================================================================
# One row per at-risk customer-month, every feature block, forward labels at
# each horizon, and a rolling-origin fold id. v5 trains on this and nothing
# else, so the leakage decisions are made HERE, once, in code that is read
# rather than in a modelling notebook where they are easy to lose:
#
#   - static attributes are read as of m - ATTR_LAG_M
#   - peer ranks are computed within-month, so no future month contributes
#   - y{h} looks forward only; rows whose horizon runs past the panel end
#     are marked scorable=0 rather than dropped, so the censoring is visible

mm = wl        # LAG_COLS were computed in section 6 on the unfiltered panel

KEEP = (["cust_pwr_id", "ym", "m_idx", "peer_key", "peer_level", "naics2", "naics_status",
         "size_dec", "bal_live", "risk_score", "rk", "event_m", "months_to_end"] +
        [f"y{h}" for h in HORIZONS] + LAG_COLS +
        [c for c in ALL_FEATS if c in mm.columns] +
        [c for c in TREND_COLS if c in mm.columns] +
        [c for c in BAL_COLS if c in mm.columns] +
        [c for c in PR_COLS if c in mm.columns])
KEEP = list(dict.fromkeys([c for c in KEEP if c in mm.columns]))

mm = mm.select(*KEEP)
for h in HORIZONS:
    mm = mm.withColumn(f"scorable_{h}", (F.col("months_to_end") >= h).cast("int"))

# rolling-origin folds: expanding train, one-period test, on m_idx
N_FOLDS = 4
fold_edges = np.linspace(M_MIN + MIN_HIST_M + 6, M_MAX, N_FOLDS + 1).astype(int)
fexpr = F.lit(None).cast("int")
for i in range(N_FOLDS):
    fexpr = F.when(F.col("m_idx").between(int(fold_edges[i]) + 1, int(fold_edges[i + 1])),
                   F.lit(i)).otherwise(fexpr)
mm = mm.withColumn("fold", fexpr)

mm.write.mode("overwrite").partitionBy("ym").parquet(hp("model_matrix"))
_mm = spark.read.parquet(hp("model_matrix"))

disp(_mm.groupBy("fold").agg(F.count("*").alias("rows"),
        F.min("ym").alias("ym_min"), F.max("ym").alias("ym_max"),
        *[F.sum(f"y{h}").alias(f"pos_y{h}") for h in HORIZONS]).orderBy("fold"),
     title="11a &middot; model_matrix written — rolling-origin folds", save="v4_model_matrix_folds")

kv({"path": hp("model_matrix"),
    "rows": f"{_mm.count():,}",
    "columns": len(_mm.columns),
    "raw features": len([c for c in ALL_FEATS if c in _mm.columns]),
    "trend features": len([c for c in TREND_COLS if c in _mm.columns]),
    "peer ranks": len([c for c in PR_COLS if c in _mm.columns]),
    "static (lagged " + str(ATTR_LAG_M) + "m)": len([c for c in LAG_COLS if c in _mm.columns]),
    "horizons": ", ".join(str(h) for h in HORIZONS),
    "folds": N_FOLDS},
   title="11b &middot; Handoff to v5", save="v4_model_matrix_summary")

disp(pd.DataFrame(FINDINGS), title="11c &middot; Findings register", n=40)
if WARNINGS:
    disp(pd.DataFrame({"warning": WARNINGS}),
         title="11d &middot; Warnings raised during this run — read every one before briefing",
         n=40, save="v4_warnings")

---

## After this run

Read in this order.

1. **§1c and §2c — the new-entity rebuild.** If `middle_over_last6` was well
   above 1 in 1c and the trailing-window CV in 2c is materially lower, the
   burn-in was masking a trend and every v3 statement involving `cpty_new_out`
   or `fin_new_out` was made on a calendar-drifting feature.

2. **§7b — the censoring test.** If the H18 lead is longer than the H12 lead,
   the briefed 4-month figure is a floor, not a measurement, and the number to
   brief changes. If H18 agrees, the 4 months is real and the question closes.

3. **§8c — the two earliest signals finally have operating points.** This is
   v3 §7.1, the first item in its sequenced plan. It may reorder the whole
   recommendation.

4. **§10a — precision@k.** The comparison that matters is
   `true_positives_per_month` at k=250, horizon 6, against v3's best six-month
   option of 17 departures in a list of 130. The target in the handoff was
   70–80 in 250.

5. **§10b — stability across periods.** This is the first out-of-time read this
   work has ever had. A score that swings by more than a few points here will
   not survive a rolling-origin fold, and a model trained on it will look far
   better in development than in production.

6. **§11d — the warnings.** Every optional column resolves or warns. A missing
   `opened_dt` means tenure is absent, which removes one of the strongest
   static predictors; a missing NAICS column collapses the peer group to
   size-decile and guts the largest false-positive lever.

**Then v5.** Gradient-boosted discrete-time hazard on `model_matrix`, trained
per horizon, rolling-origin on `fold`, evaluated on precision@k and
dollar-weighted precision@k — the same metrics as §10 so the comparison is
direct. Report the incremental gain in four blocks: deposit only → + static
attributes → + payment features → + peer ranks and trend. A model that does not
clearly beat the untrained rank score in §10 has not earned its
maintenance cost.

**Still not addressed, and still worth doing.**

- **Destination-institution classification** (v3 §8.3 row 2). `cpty_fin_entity_name`
  is extracted and only counted. Separating competitor treasury operations from
  vendors, utilities and processors is the only route to the
  competitor-vs-contraction question in §7.7, and it needs a classification
  pass that does not fit in this notebook.
- **Relationship grain** (`rltn_pwr_id`). 49,294 relationships against 120,457
  customers, and Treasury sells to the relationship. The unit-of-analysis
  decision is still open.
- **Graph features.** Deliberately out of scope here. When they are tested, do
  it on the payment-visible B2B slice as its own ablation rung rather than
  diluting them across a book that is mostly graph-invisible.

---

*Internal — PNC Treasury Management, Data Science*